In [1]:
name="dvsgenaib3"
mode="online"

In [2]:
print(f"dvstechnology is starting a new batch {name} and its mode is {mode}")

dvstechnology is starting a new batch dvsgenaib3 and its mode is online


In [7]:
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [8]:
!pip install openai

In [9]:
def get_completion(prompt, model="gpt-4o"):
    """
    Sends a single prompt to the specified OpenAI chat model and returns the response.

    Args:
        prompt (str): The user input or query to send to the model.
        model (str): The name of the OpenAI model to use (default is "gpt-4o").

    Returns:
        str: The assistant's textual response from the model.
    """
    from openai import OpenAI
    openai_client = OpenAI()
    # Construct the message list with a single user message
    messages = [{"role": "user", "content": prompt}]

    # Make the API call to OpenAI's chat completion endpoint
    response = openai_client.chat.completions.create(
        model=model,          # Specify which model to use (e.g., gpt-4o, gpt-4, gpt-3.5-turbo)
        messages=messages,    # The message history to provide context (only one user message here)
        temperature=0         # Controls randomness: 0 = deterministic, 1 = creative/random
    )

    # Extract and return the generated response content from the first choice
    return response.choices[0].message.content

#### Tactic1: Use delimeters


In [3]:
pharma_text = """
CTX-214 Phase III trial evaluated efficacy and safety in 400 patients with Type 2 Diabetes.
The study spanned 15 global sites and lasted 24 weeks.
Primary endpoint: Reduction in HbA1c.
Results: Statistically significant improvement with no serious adverse events reported.
"""

pharma_text1 = """
CTX-215 Phase III trial evaluated efficacy and safety in 400 patients with Type 2 Diabetes.
The study spanned 15 global sites and lasted 24 weeks.
Primary endpoint: Reduction in HbA1c.
Results: Statistically significant improvement with no serious adverse events reported.
"""

pharma_text2 = """
CTX-216 Phase III trial evaluated efficacy and safety in 400 patients with Type 2 Diabetes.
The study spanned 15 global sites and lasted 24 weeks.
Primary endpoint: Reduction in HbA1c.
Results: Statistically significant improvement with no serious adverse events reported.
"""


In [4]:
prompt = f"""
You are a clinical research analyst.
Summarize the information between the <trial> tags below into a single sentence
focusing on: drug/trial name, indication, patient count, duration, outcome, and safety.

---
<trial>
{pharma_text}
</trial>
---
---
<trial>
{pharma_text1}
</trial>
---
---
<trial>
{pharma_text2}
</trial>
---
"""
print("Input Prompt:",prompt)

Input Prompt: 
You are a clinical research analyst.
Summarize the information between the <trial> tags below into a single sentence
focusing on: drug/trial name, indication, patient count, duration, outcome, and safety.

---
<trial>

CTX-214 Phase III trial evaluated efficacy and safety in 400 patients with Type 2 Diabetes.
The study spanned 15 global sites and lasted 24 weeks.
Primary endpoint: Reduction in HbA1c.
Results: Statistically significant improvement with no serious adverse events reported.

</trial>
---
---
<trial>

CTX-215 Phase III trial evaluated efficacy and safety in 400 patients with Type 2 Diabetes.
The study spanned 15 global sites and lasted 24 weeks.
Primary endpoint: Reduction in HbA1c.
Results: Statistically significant improvement with no serious adverse events reported.

</trial>
---
---
<trial>

CTX-216 Phase III trial evaluated efficacy and safety in 400 patients with Type 2 Diabetes.
The study spanned 15 global sites and lasted 24 weeks.
Primary endpoint: R

In [10]:
print("**************************")
response = get_completion(prompt)
print("Completion:",response)

**************************
Completion: The CTX-214, CTX-215, and CTX-216 Phase III trials each evaluated the efficacy and safety of their respective drugs in 400 patients with Type 2 Diabetes over 24 weeks across 15 global sites, achieving statistically significant improvements in HbA1c reduction without any serious adverse events reported.


####Tactic 2: Ask for structured output
# Json, HTML

In [11]:
output_format = "json"
prompt = f"""
Generate a list of three fictional pharmaceutical drugs \
along with their manufacturers and therapeutic areas.
Provide them in {output_format} format with the following keys:
drug_id, name, manufacturer, therapeutic_area.
"""
response = get_completion(prompt)
print(response)

```json
[
    {
        "drug_id": "RX001",
        "name": "CardioRelief",
        "manufacturer": "HealthGenix Pharmaceuticals",
        "therapeutic_area": "Cardiovascular"
    },
    {
        "drug_id": "RX002",
        "name": "NeuroCalm",
        "manufacturer": "NeuroPharm Solutions",
        "therapeutic_area": "Neurology"
    },
    {
        "drug_id": "RX003",
        "name": "GastroEase",
        "manufacturer": "DigestiveCare Inc.",
        "therapeutic_area": "Gastroenterology"
    }
]
```


###Tactic 3: Ask the model to check whether conditions are satisfied

In [12]:
text_1 = f"""
The clinical trial began with a screening visit to determine subject eligibility,
including medical history and laboratory tests. Upon passing the screening,
participants entered a 2-week washout period where any previous medications were discontinued.
After the washout, subjects received the first dose of the investigational drug under supervision.
Vital signs were monitored every 30 minutes for the next 4 hours.
Subjects then returned for follow-up visits on Days 7, 14, and 28 to assess safety and efficacy.
"""

prompt = f"""
You will be provided with clinical procedure text delimited by triple quotes.
If it contains a sequence of procedural steps, re-write them in the following format:

Step 1 - ...
Step 2 - ...
…
Step N - …

If the text does not contain a sequence of steps or instructions, \
then simply write \"No steps provided.\"

\"\"\"{text_1}\"\"\"
"""
response = get_completion(prompt)
print("Completion for Text 1:")
print(response)

Completion for Text 1:
Step 1 - Begin with a screening visit to determine subject eligibility, including medical history and laboratory tests.

Step 2 - Upon passing the screening, enter a 2-week washout period where any previous medications are discontinued.

Step 3 - After the washout, receive the first dose of the investigational drug under supervision.

Step 4 - Monitor vital signs every 30 minutes for the next 4 hours.

Step 5 - Return for follow-up visits on Days 7, 14, and 28 to assess safety and efficacy.


In [13]:
text_2 = f"""
The clinical trial results were promising, with a significant reduction in blood pressure
observed in the treatment group. Participants reported minimal side effects, and
overall tolerability was high. The study was conducted across multiple sites and
included a diverse patient population. Investigators noted improvements in adherence
and patient-reported outcomes, especially among those who had previously shown resistance
to standard antihypertensive therapies.
"""

prompt = f"""
You will be provided with text delimited by triple quotes.
If it contains a sequence of instructions, \
re-write those instructions in the following format:

Step 1 - ...
Step 2 - …
…
Step N - …

If the text does not contain a sequence of instructions, \
then simply write "No steps provided."

\"\"\"{text_2}\"\"\"
"""
response = get_completion(prompt)
print("Completion for Text 2:")
print(response)

Completion for Text 2:
No steps provided.
